## <b><u>Infrared.city-1</u>: Tree detection</b>
<i>created by Nina Salnikow</i>
### <b>Introduction</b>

As urban areas are expanding and global temperatures rise due to climate change, being able to understand how trees can help cool cities has become increasingly important. Furthermore, trees reduce the risk of flooding in cities and promote health and wellbeing. Our project aims at taking the first step in the direction of better tree coverage in urban areas by creating a model which is able to predict tree locations using satellite imagery of densely populated areas. 

### <b>Notebook structure 4</b>


&nbsp;&nbsp;&nbsp;<b>1. Getting total bounds of each cities</b><br>
&nbsp;&nbsp;&nbsp;<b>2. Documentation of fetching sentinel-2 test data</b><br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<b>2.1 Prague</b><br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<b>2.2 Budapest</b><br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<b>2.3 Warsaw</b><br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<b>2.4 Milan</b><br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<b>2.5 Athens</b><br>

### <b><u>New</u> shared folder structure</b>
/home/jovyan/ideas-dslab-group1-shared/<br>
├── <u>raw data/</u> (backup)<br>
│&ensp;&ensp;&ensp;      ├── baumkataster data/ <br>
│&ensp;&ensp;&ensp;   ├── <s>sentinel data/</s> -> already cleaned<br>
│&ensp;&ensp;&ensp;   ├── osm data/ <br>
│&ensp;&ensp;&ensp;   └── vienna context data/<br>
└── <u>cleaned data/</u><br>
&ensp;&ensp;&ensp; &ensp;├── baumkataster data/ <br>
&ensp;&ensp;&ensp;&ensp;   ├── sentinel data/ <br>
&ensp;&ensp;&nbsp;&nbsp;&nbsp;&nbsp;     │&ensp;&ensp;&ensp; └── <b>data for database/ -> added 5 cities ( each 3 seasons) mentioned below</b><br>
&ensp;&ensp;&ensp;&ensp;   ├── osm data/ <br>
&ensp;&ensp;&ensp;&ensp;   └── vienna context data/<br>
<br>

### <b>New Citys Test data overview</b>

<blockquote>
Our data coaches from <b><a hre="https://infrared.city/knowledge-base/available-models/">Infrared.city</a></b> are particularly interested in cities with strong urban heat.<br>
</blockquote>

<i>I will collect Sentinel-2 satellite imagery to generate tree-location databases for selected European cities. Open tree cadastre data is often unavailable and incomplete (e.g. no suitable dataset was found for Budapest, while only a 2020 tree cadastre was available for <a href="https://data.europa.eu/data/datasets/https-kod-brno-cz-nkod-dataset-1c69f0c46f624344a22535860cfa5a0b_0-ttl?locale=en">Brno with no translation to englisch</a> ).</i><br>
<br><b>Therefore, we choose following cities:</b>

- <u><b>Prague</b></u>: similar to Vienna

- <u><b>Budapest</b></u>: limited tree data available

- <u><b>Warsaw</b></u>: large Central European capital

- <u><b>Milan</b></u>: warmer Southern European city

- <u><b>Athens</b></u>: hot Mediterranean city


<u>Satellite image data scraper source:</u>

- <b><a href="https://code.earthengine.google.com/">Sentinel-2 Satellite Images</a></b> <i>Multispectral satellite imagery including 10 m resolution bands (GeoTIFF)</i><br><br>

### <b>1. Getting total bounds of each cities</b>

In [1]:
!pip install geopy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [geopy]


In [2]:
import time #just in case, I dont wanna be rude and overload c:
#https://geopy.readthedocs.io/en/stable/
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="gee_bbox_loop")
staedte = ["Prague", "Budapest", "Warsaw", "Milan", "Athens"]
for stadt in staedte:
    loc = geolocator.geocode(stadt)
    b = loc.raw["boundingbox"]
    print(f"  '{stadt.lower()}': [{b[2]}, {b[0]}, {b[3]}, {b[1]}],")
    #https://stackoverflow.com/questions/35167624/how-to-make-geopy-requests
    time.sleep(4) #we are nice, pls everyone do the same when you scrape thxxx

  'prague': [14.2244355, 49.9419006, 14.7067867, 50.1774302],
  'budapest': [18.9251057, 47.3496899, 19.3349258, 47.6131468],
  'warsaw': [20.8516882, 52.0978497, 21.2711512, 52.3681531],
  'milan': [9.0408867, 45.3867381, 9.2781103, 45.5358482],
  'athens': [-83.5374696, 33.8480209, -83.2408342, 34.0394660],


### <b>2. Documentation of fetching sentinel-2 test data</b>

>I will again use following adjusted code to scrape all 5 cities for our database.

```js

//output of our total bounds, that we calculated above 
//if the bounds are correct: google earth will show you on the map below the correct city name -> nice visual verification ngl
var cities = {'prague': [14.2244355, 49.9419006, 14.7067867, 50.1774302], 
  'budapest': [18.9251057, 47.3496899, 19.3349258, 47.6131468], 
  'warsaw': [20.8516882, 52.0978497, 21.2711512, 52.3681531],
  'milan': [9.0408867, 45.3867381, 9.2781103, 45.5358482],
  'athens': [-83.5374696, 33.8480209, -83.2408342, 34.0394660]};
var cityName = 'linz'; // just change the city name to reuse the script
var aoi = ee.Geometry.Rectangle(cities[cityName]);

//we take april=spring, august=summer and nov=autumn like the previous project
//https://gis.stackexchange.com/questions/363857/using-date-list-to-filter-image-collection-in-google-earth-engine (accessed 31.03 26)
[{n:'apr', s:'2025-04-01', e:'2025-04-30'},
 {n:'aug', s:'2025-08-01', e:'2025-08-31'},
 {n:'nov', s:'2025-11-01', e:'2025-11-30'}].forEach(function(m) {
  //filtering sen-2 data
  var s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') //SR_HARMONIZED' = corrected data :o
    .filterBounds(aoi)
    .filterDate(m.s, m.e)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20));
  var image = s2.median().clip(aoi); //make avg with median=more robust cloud free pic, recommendation from stackoverflow
  
  //checking if the year even has pics  
  print(cityName + ' ' + m.n + ' image count', s2.size()); 
  
  //scale bands first, then calculate NDVI/EVI/SAVI like prev project <- change this part as we had EVI calc error last time
  var refl = image.select(['B2','B3','B4','B8']).divide(10000);
  var ndvi = refl.normalizedDifference(['B8','B4']).rename('NDVI');
  var evi = refl.expression('2.5*((N-R)/(N+6*R-7.5*B+1))', {N: refl.select('B8'), R: refl.select('B4'), B: refl.select('B2')}).rename('EVI');
  var savi = refl.expression('1.5*((N-R)/(N+R+0.5))', {N: refl.select('B8'), R: refl.select('B4')}).rename('SAVI');
  var stack = refl.addBands([ndvi, evi, savi]).toFloat();
     
  //https://stackoverflow.com/questions/52450596/exporting-images-in-an-image-collection-from-gee (accessed 31.03.26)
  Export.image.toDrive({image: stack,
    description: 'S2_'+cityName+'_'+ m.n, folder: 'EarthEngineExports', //also has to define folder, else it was confused -> still better than copernicus
    scale: 10, region: aoi,
    fileFormat: 'GeoTIFF', maxPixels: 1e13 //got an error, had to max up the resolution
  });
     
  //https://developers.google.com/earth-engine/apidocs/map-addlayer (accessed 31.03.26)
  Map.addLayer(image,{bands:['B4','B3','B2'], min:0, max:3000}, cityName + '_' + m.n,false);
});

Map.centerObject(aoi, 12);
Map.addLayer(aoi, {}, cityName + '_AOI');

```

<br>

#### <b>Here are the outputs of following codeline for every city:</b><br>
<blockquote><code>print(cityName + ' ' + m.n + ' image count', s2.size());</code><br>

<i>Note: save files it with <u>EPSG:4326</u>!!</i>
</blockquote>



### <b>2.1 Prague</b>

prague apr image count: <code>5</code><br>
prague aug image count: <code>7</code><br>
prague oct image count: <code>2</code> (<i>-> for November there was only 1</i>)

### <b>2.2 Budapest</b>

budapest apr image count: <code>12</code><br>
budapest aug image count: <code>19</code><br>
budapest nov image count: <code>8</code> (<i>-> for October there was only 5</i>)

### <b>2.3 Warsaw</b>

warsaw apr image count: <code>23</code><br>
warsaw aug image count: <code> 19</code><br>
warsaw oct image count: <code>9 </code>(<i>-> for November there was only 7</i>)


### <b>2.4 Milan </b>

milan apr image count: <code>3</code><br>
milan aug image count: <code>10</code><br>
milan nov image count: <code>5</code> (<i>-> same for Oct</i>)

### <b>2.5 Athens</b>

athens apr image count: <code>2</code> (<i>-> low but I only want to change month if we have like 0-1 images</i>)<br>
athens july image count: <code>4</code> (<i>-> there was none for august for me :c</i>)<br>
athens nov image count: <code>5</code> (<i>-> only 3 in oct</i>)<br>


In [3]:
!pip install rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.5/37.5 MB 113.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [rasterio]2/3 [rasterio]


In [4]:
import rasterio #checking just in case, maybe i forgot to input: EPSG:4326, then i have to rescrape

print(rasterio.open("/home/jovyan/ideas-dslab-group1-shared/cleaned data/sentinel data/data for database/S2_prague_apr_25.tif").crs)
print(rasterio.open("/home/jovyan/ideas-dslab-group1-shared/cleaned data/sentinel data/data for database/S2_prague_aug_25.tif").crs)
print(rasterio.open("/home/jovyan/ideas-dslab-group1-shared/cleaned data/sentinel data/data for database/S2_prague_oct_25.tif").crs)

print(rasterio.open("/home/jovyan/ideas-dslab-group1-shared/cleaned data/sentinel data/data for database/S2_budapest_apr_25.tif").crs)
print(rasterio.open("/home/jovyan/ideas-dslab-group1-shared/cleaned data/sentinel data/data for database/S2_budapest_aug_25.tif").crs)
print(rasterio.open("/home/jovyan/ideas-dslab-group1-shared/cleaned data/sentinel data/data for database/S2_budapest_nov_25.tif").crs)

print(rasterio.open("/home/jovyan/ideas-dslab-group1-shared/cleaned data/sentinel data/data for database/S2_warsaw_apr_25.tif").crs)
print(rasterio.open("/home/jovyan/ideas-dslab-group1-shared/cleaned data/sentinel data/data for database/S2_warsaw_aug_25.tif").crs)
print(rasterio.open("/home/jovyan/ideas-dslab-group1-shared/cleaned data/sentinel data/data for database/S2_warsaw_oct_25.tif").crs)

print(rasterio.open("/home/jovyan/ideas-dslab-group1-shared/cleaned data/sentinel data/data for database/S2_milan_apr_25.tif").crs)
print(rasterio.open("/home/jovyan/ideas-dslab-group1-shared/cleaned data/sentinel data/data for database/S2_milan_aug_25.tif").crs)
print(rasterio.open("/home/jovyan/ideas-dslab-group1-shared/cleaned data/sentinel data/data for database/S2_milan_nov_25.tif").crs)

print(rasterio.open("/home/jovyan/ideas-dslab-group1-shared/cleaned data/sentinel data/data for database/S2_athens_apr_25.tif").crs)
print(rasterio.open("/home/jovyan/ideas-dslab-group1-shared/cleaned data/sentinel data/data for database/S2_athens_july_25.tif").crs)
print(rasterio.open("/home/jovyan/ideas-dslab-group1-shared/cleaned data/sentinel data/data for database/S2_athens_nov_25.tif").crs)

EPSG:4326
EPSG:4326
EPSG:4326
EPSG:4326
EPSG:4326
EPSG:4326
EPSG:4326
EPSG:4326
EPSG:4326
EPSG:4326
EPSG:4326
EPSG:4326
EPSG:4326
EPSG:4326
EPSG:4326


<blockquote>Perfect, all images are EPSG:4326.</blockquote>